# Student Performance Prediction Pipeline (used CatBoost)
Multi-course training, per-course outlier handling, unseen-course validation and artifact export.


File paths, configuration and features

In [15]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

TRAIN_PATH = Path("generated_canvas_export.csv")
REAL_PATH = Path("canvas_api_export.csv")
MODEL_PATH = Path("student_predictor.cbm")
META_PATH = Path("model_metadata.json")
PREDICTIONS_PATH = Path("predictions_output.csv")

TARGET_COL = "total_score"
FEATURE_COLS = [
    "course_files_count",
    "online_assignments_count",
    "due_online_assignments_count",
    "offline_assignments_count",
    "clipped_time_seconds",
    "relative_time_ratio",
    "relative_inactivity_days",
    "submitted_count",
    "submission_rate",
    "late_count",
    "missing_count",
    "total_attempts",
    "student_comments",
    "avg_hours_before_deadline",
]

Data clearing and risk tiers

In [16]:
def clip_iqr(s: pd.Series) -> pd.Series:
    """Cap extreme high outliers using 1.5 * IQR."""
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr if iqr > 0 else s.max()
    return s.clip(upper=upper)


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Clean Canvas time anomalies and compute course-relative features."""
    data = df.copy()

    # 1. Cap AFK browser-tab time outliers per course
    data["clipped_time_seconds"] = data.groupby("course_id")["total_time_seconds"].transform(clip_iqr)

    # 2. Normalize time and inactivity within each course
    median_time = data.groupby("course_id")["clipped_time_seconds"].transform("median").replace(0, 1)
    data["relative_time_ratio"] = (data["clipped_time_seconds"] / median_time).round(2)

    min_inactive = data.groupby("course_id")["days_since_last_activity"].transform("min")
    data["relative_inactivity_days"] = (data["days_since_last_activity"] - min_inactive).round(2)

    # 3. Treat deadline delta as missing when no tasks were submitted
    data.loc[data["submitted_count"] == 0, "avg_hours_before_deadline"] = np.nan

    return data


def get_risk_level(score: float) -> str:
    if score < 50.0:
        return "High Risk"
    return "Medium Risk" if score < 70.0 else "Low Risk"

Load data and prepare model

In [17]:
df = engineer_features(pd.read_csv(TRAIN_PATH))

# Split by course_id to validate generalization on unseen courses
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(df, df[TARGET_COL], groups=df["course_id"]))

X_train, y_train = df.iloc[train_idx][FEATURE_COLS], df.iloc[train_idx][TARGET_COL]
X_val, y_val = df.iloc[val_idx][FEATURE_COLS], df.iloc[val_idx][TARGET_COL]

model = CatBoostRegressor(iterations=1000, depth=5, learning_rate=0.04, l2_leaf_reg=5.0, loss_function="RMSE", eval_metric="MAE", random_seed=42, early_stopping_rounds=50, verbose=100,)
model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)


0:	learn: 27.6987217	test: 31.9200014	best: 31.9200014 (0)	total: 1.3ms	remaining: 1.3s
100:	learn: 4.3997780	test: 6.0871890	best: 6.0871890 (100)	total: 112ms	remaining: 998ms
200:	learn: 3.7135054	test: 5.5704599	best: 5.5700131 (199)	total: 204ms	remaining: 810ms
Stopped by overfitting detector  (50 iterations wait)

bestTest = 5.539020367
bestIteration = 228

Shrink model to first 229 iterations.


CatBoostRegressor(depth=5, early_stopping_rounds=50, eval_metric='MAE', iterations=1000, l2_leaf_reg=5.0, learning_rate=0.04, loss_function='RMSE', random_seed=42, verbose=100)

Making prediction, score difference and save output

In [19]:
val_preds = np.clip(model.predict(X_val), 0.0, 100.0)
metrics = {
    "val_mae": round(float(mean_absolute_error(y_val, val_preds)), 2),
    "val_rmse": round(float(np.sqrt(mean_squared_error(y_val, val_preds))), 2),
    "val_r2": round(float(r2_score(y_val, val_preds)), 4),
    "best_iteration": int(model.get_best_iteration()),
}
print("Validation metrics:", metrics)

if REAL_PATH.exists():
    real_df = engineer_features(pd.read_csv(REAL_PATH))
    real_df["predicted_score"] = np.clip(model.predict(real_df[FEATURE_COLS]), 0.0, 100.0).round(1)
    real_df["score_difference"] = (real_df[TARGET_COL] - real_df["predicted_score"]).round(1)
    real_df["risk_level"] = real_df["predicted_score"].apply(get_risk_level)

    cols = ["course_code", "student_name", "total_time_seconds", TARGET_COL, "predicted_score", "risk_level"]
    print("\nReal Canvas validation:")
    print(real_df[cols].to_string(index=False))

    real_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

Validation metrics: {'val_mae': 5.52, 'val_rmse': 7.16, 'val_r2': 0.9603, 'best_iteration': 228}

Real Canvas validation:
         course_code student_name  total_time_seconds  total_score  predicted_score risk_level
202610:CSS4004-ENG-1    Student 1                2921         93.0             82.7   Low Risk
202610:CSS4004-ENG-1    Student 2                 309         70.0             71.0   Low Risk
202610:CSS4004-ENG-1    Student 3                 243         20.0             28.4  High Risk
202610:CSS4004-ENG-1    Student 4                 120          0.0             15.1  High Risk
202610:PSY2001-RUS-1    Student 1                 231         29.0             31.5  High Risk
202610:PSY2001-RUS-1    Student 2                 187         75.0             76.9   Low Risk
202610:PSY2001-RUS-1    Student 3                 201         92.0             76.8   Low Risk
202610:PSY2001-RUS-1    Student 4                   0         53.0             70.3   Low Risk


Save model and metadata

In [20]:
model.save_model(str(MODEL_PATH))

metadata = {
    "model_version": "1.0.0",
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "target_column": TARGET_COL,
    "feature_columns": FEATURE_COLS,
    "risk_thresholds": {"high_risk_below": 50.0, "medium_risk_below": 70.0},
    "metrics": metrics,
}

META_PATH.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved {MODEL_PATH} and {META_PATH}")

Saved student_predictor.cbm and model_metadata.json
